# MuSiQue · build the questions (local)

Writes `questions.pkl`: MuSiQue 2-hop questions, each with its 2 supporting paragraphs and 16 distractor
paragraphs turned into facts by Llama-3.1-70B, in a fixed nested order. Every notebook in this folder reads it.
The steps and filters are described at the top of `common/build.py`. Cost: about $1 of 70B calls.

**Run once.** If `questions.pkl` exists, this notebook only shows it; set `REBUILD = True` to build a new set (then
move `results/` away: answers to an old question set are never mixed with a new one).

In [ ]:
import os, pickle, sys
sys.path.insert(0, "../common")

DATASET = "MuSiQue"
N_QUESTIONS = 200
K_MAX = 16            # distractor paragraphs per question (k = 0, 4, 8, 16)
REBUILD = False

In [ ]:
if os.path.exists("questions.pkl") and not REBUILD:
    print("questions.pkl already exists: set REBUILD = True to build a new set")
else:
    from build import build
    build(DATASET, N_QUESTIONS, K_MAX)

In [ ]:
from collections import Counter
questions = pickle.load(open("questions.pkl", "rb"))
print(f"{len(questions)} questions · sources {dict(Counter(q['source_split'] for q in questions))} · "
      f"{len({q['answer'] for q in questions})} distinct answers")
for q in questions[:5]:
    supporting = sum(len(p["facts"]) for p in q["layout"] if p["is_supporting"])
    distractor = sum(len(p["facts"]) for p in q["layout"] if not p["is_supporting"])
    print(f"- {q['question']}  →  {q['answer']}   ({supporting} supporting facts, {distractor} in distractors)")